In [6]:
# Environment: Python 3.10 (ml310)
# XGBoost for Insurance Type Prediction
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

nhanes_clean_adult_add = pd.read_csv("nhanes_clean_adult_add.csv")

ins_feature_cols = [
    'age',
    'gender',
    'race',
    'education',
    'income_poverty_ratio',
    'marital_status',
    'household_size',
    'is_employed',
    # new features
    'smoking_current',
    'diabetes_self_report',
    'bmi',
    'sbp',
    'dbp',
    'a1c',
    'sleep_hours'
]

ins_target_col = 'insurance_type'

X_ins = nhanes_clean_adult_add[ins_feature_cols].copy()
y_ins = nhanes_clean_adult_add[ins_target_col].copy()

data_ins = pd.concat([X_ins, y_ins], axis=1).dropna(subset=ins_feature_cols + [ins_target_col])
X_ins = data_ins[ins_feature_cols]
y_ins = data_ins[ins_target_col]

# y encode to numeric data
from sklearn.preprocessing import LabelEncoder
le_ins = LabelEncoder()
y_ins_encoded = le_ins.fit_transform(y_ins)   # -> 0,1,2,3,4

X_ins_train, X_ins_test, y_ins_train, y_ins_test = train_test_split(
    X_ins, y_ins_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_ins_encoded
)

cat_ins_features = [
    'gender',
    'race',
    'education',
    'marital_status',
    'is_employed',
    'smoking_current',
    'diabetes_self_report'
]

num_ins_features = [
    'age',
    'income_poverty_ratio',
    'household_size',
    'bmi',
    'sbp',
    'dbp',
    'a1c',
    'sleep_hours'
]

preprocess_ins = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_ins_features),
        ('num', 'passthrough', num_ins_features)
    ]
)

# Define XGBoost Classifier
xgb_ins = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Pipeline setup
xgb_ins_pipeline = Pipeline(steps=[
    ('preprocess', preprocess_ins),
    ('clf', xgb_ins)
])

#StratifiedKFold Cross Validation
cv_ins = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# K=5 Cross Validation Result for Preliminary Analysis
base_scores_ins = cross_val_score(
    xgb_ins_pipeline,
    X_ins_train,
    y_ins_train,
    cv=cv_ins,
    scoring='accuracy',
    n_jobs=-1
)

print("XGBoost (Insurance) - 5-fold CV accuracy:", base_scores_ins)
print("Mean CV accuracy:", base_scores_ins.mean(), "\n")

# #GridSearchCV for Estimator Optimization
param_grid_ins = {
    'clf__n_estimators': [200, 300],
    'clf__max_depth': [3, 4, 5],
    'clf__learning_rate': [0.05, 0.1],
    'clf__subsample': [0.8, 1.0],
    'clf__colsample_bytree': [0.8, 1.0]
}

grid_ins = GridSearchCV(
    estimator=xgb_ins_pipeline,
    param_grid=param_grid_ins,
    cv=cv_ins,
    scoring='accuracy',
    n_jobs=-1
)

grid_ins.fit(X_ins_train, y_ins_train)


print("Best CV accuracy (Insurance, XGB):", grid_ins.best_score_)
print("Best params:")
for k, v in grid_ins.best_params_.items():
    print(f"  {k}: {v}")

#Best Pipeline
best_xgb_ins = grid_ins.best_estimator_


# ========== 8. Test 集最终评估 ==========
# best_xgb_ins.fit(X_ins_train, y_ins_train)
# y_ins_pred = best_xgb_ins.predict(X_ins_test)

# 把数字标签还原成原来的保险类型字符串
# y_ins_test_labels = le_ins.inverse_transform(y_ins_test)
# y_ins_pred_labels = le_ins.inverse_transform(y_ins_pred)

# print("\nTest accuracy (Insurance, XGB):", accuracy_score(y_ins_test, y_ins_pred))
# print("\nClassification report (label names):")
# print(classification_report(y_ins_test_labels, y_ins_pred_labels))

XGBoost (Insurance) - 5-fold CV accuracy: [0.63678161 0.62911877 0.65107362 0.65107362 0.62039877]
Mean CV accuracy: 0.6376892790823402 

Best CV accuracy (Insurance, XGB): 0.6488819547281575
Best params:
  clf__colsample_bytree: 0.8
  clf__learning_rate: 0.05
  clf__max_depth: 3
  clf__n_estimators: 200
  clf__subsample: 0.8


In [11]:
#XGboost for Hypertension ： Normal(+High-normal), Stage 1, Stage 2 with SMOTE

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

def classify_hypertension_stage(row):
    sbp = row['sbp']
    dbp = row['dbp']

    if pd.isna(sbp) or pd.isna(dbp):
        return None  

    # Stage 2
    if sbp >= 140 or dbp >= 90:
        return "Stage 2"
    # Stage 1
    elif 130 <= sbp < 140 or 80 <= dbp < 90:
        return "Stage 1"
    # Normal + High-normal
    else:
        return "Normal"
        
nhanes_clean_adult_add['hypertension_stage'] = nhanes_clean_adult_add.apply(
    classify_hypertension_stage, axis=1
)

hyp_feature_cols = [
    'age',
    'gender',
    'race',
    'education',
    'income_poverty_ratio',
    'marital_status',
    'household_size',
    'is_employed',
    'smoking_current',
    'diabetes_self_report',
    'bmi',
    'a1c',
    'sleep_hours',
    'insurance_type'
]

hyp_target_col = 'hypertension_stage'

X_hyp = nhanes_clean_adult_add[hyp_feature_cols].copy()
y_hyp = nhanes_clean_adult_add[hyp_target_col].copy()

data_hyp = pd.concat([X_hyp, y_hyp], axis=1).dropna(subset=hyp_feature_cols + [hyp_target_col])
X_hyp = data_hyp[hyp_feature_cols]
y_hyp = data_hyp[hyp_target_col]

# y encode to numeric data
from sklearn.preprocessing import LabelEncoder
le_ins = LabelEncoder()
y_hyp_encoded = le_ins.fit_transform(y_hyp)   # -> 0,1,2,3,4

X_hyp_train, X_hyp_test, y_hyp_train, y_hyp_test = train_test_split(
    X_hyp, y_hyp_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_hyp_encoded
)

cat_hyp_features = [
    'gender',
    'race',
    'education',
    'marital_status',
    'is_employed',
    'smoking_current',
    'diabetes_self_report',
    'insurance_type'
]

num_hyp_features = [
    'age',
    'income_poverty_ratio',
    'household_size',
    'bmi',
    'a1c',
    'sleep_hours'
]


preprocess_hyp = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_hyp_features),
        ('num', 'passthrough', num_hyp_features)
    ]
)

# Define XGBoost Classifier
xgb_hyp = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Preprocess → SMOTE → XGB
xgb_hyp_pipeline = Pipeline(steps=[
    ('preprocess', preprocess_hyp),
    ('smote', SMOTE(random_state=42)),
    ('clf', xgb_hyp)
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 5-fold CV Baseline
base_scores_hyp = cross_val_score(
    xgb_hyp_pipeline,
    X_hyp_train,
    y_hyp_train,
    cv=cv,
    scoring='accuracy'
)

print("XGBoost (Hypertension) - 5-fold CV accuracy:", base_scores_hyp)
print("Mean CV accuracy:", base_scores_hyp.mean(), "\n")

# GridSearchCV Optimization
param_grid_hyp = {
    'clf__n_estimators': [150, 250],
    'clf__max_depth': [3, 4, 5],
    'clf__learning_rate': [0.05, 0.1],
    'clf__subsample': [0.8, 1.0],
    'clf__colsample_bytree': [0.8, 1.0]
}

grid_hyp = GridSearchCV(
    estimator=xgb_hyp_pipeline,
    param_grid=param_grid_hyp,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_hyp.fit(X_hyp_train, y_hyp_train)

print("Best CV accuracy (Hypertension, XGB):", grid_hyp.best_score_)
print("Best params:")
for k, v in grid_hyp.best_params_.items():
    print(f"  {k}: {v}")

best_xgb_hyp = grid_hyp.best_estimator_

# 把数字标签还原成原来的保险类型字符串
# y_hyp_test_labels = le_ins.inverse_transform(y_hyp_test)
# y_hyp_pred_labels = le_ins.inverse_transform(y_hyp_pred)

# print("\nTest accuracy (Insurance, XGB):", accuracy_score(y_hyp_test, y_hyp_pred))
# print("\nClassification report (label names):")
# print(classification_report(y_hyp_test_labels, y_hyp_pred_labels))

XGBoost (Hypertension) - 5-fold CV accuracy: [0.65057471 0.651341   0.67254601 0.66027607 0.65490798]
Mean CV accuracy: 0.6579291540323907 

Best CV accuracy (Hypertension, XGB): 0.6616099005711868
Best params:
  clf__colsample_bytree: 0.8
  clf__learning_rate: 0.1
  clf__max_depth: 4
  clf__n_estimators: 150
  clf__subsample: 1.0


In [12]:
from sklearn.model_selection import cross_validate

scoring_hyp = {
    'acc': 'accuracy',
    'recall_macro': 'recall_macro',
    'recall_weighted': 'recall_weighted'
}

# Cross Validation
cv_results_hyp = cross_validate(
    xgb_hyp_pipeline,     # XGB + preprocess + SMOTE pipeline
    X_hyp_train,
    y_hyp_train,
    cv=cv,                # StratifiedKFold(5)
    scoring=scoring_hyp,
    n_jobs=-1
)

# Every fold result
print("CV accuracy:", cv_results_hyp['test_acc'])
print("CV macro recall:", cv_results_hyp['test_recall_macro'])
print("CV weighted recall:", cv_results_hyp['test_recall_weighted'])

# Mean
print("\nMean accuracy:", cv_results_hyp['test_acc'].mean())
print("Mean macro recall:", cv_results_hyp['test_recall_macro'].mean())
print("Mean weighted recall:", cv_results_hyp['test_recall_weighted'].mean())


CV accuracy: [0.65057471 0.651341   0.67254601 0.66334356 0.66027607]
CV macro recall: [0.38747793 0.38772945 0.40828389 0.41129713 0.39870957]
CV weighted recall: [0.65057471 0.651341   0.67254601 0.66334356 0.66027607]

Mean accuracy: 0.6596162705968079
Mean macro recall: 0.3986995943885069
Mean weighted recall: 0.6596162705968079


In [13]:
from sklearn.metrics import recall_score, make_scorer
from sklearn.model_selection import cross_val_score

# 假设 LabelEncoder() 之后:
# 0 = Normal
# 1 = Stage 1
# 2 = Stage 2

recall_stage1 = make_scorer(recall_score, average=None, labels=[1])
recall_stage2 = make_scorer(recall_score, average=None, labels=[2])

print("=== Stage 1 Recall (CV) ===")
cv_stage1 = cross_val_score(
    xgb_hyp_pipeline,
    X_hyp_train,
    y_hyp_train,
    cv=cv,
    scoring=recall_stage1
)
print(cv_stage1)
print("Mean Stage 1 Recall:", cv_stage1.mean())

print("\n=== Stage 2 Recall (CV) ===")
cv_stage2 = cross_val_score(
    xgb_hyp_pipeline,
    X_hyp_train,
    y_hyp_train,
    cv=cv,
    scoring=recall_stage2
)
print(cv_stage2)
print("Mean Stage 2 Recall:", cv_stage2.mean())

=== Stage 1 Recall (CV) ===
[0.11061947 0.08849558 0.09777778 0.10666667 0.07111111]
Mean Stage 1 Recall: 0.09493411996066864

=== Stage 2 Recall (CV) ===
[0.15463918 0.17525773 0.20512821 0.20512821 0.20512821]
Mean Stage 2 Recall: 0.18905630452022204
